# Model Experiment — `N-BEATS`

## Theory

**N-BEATS** (Neural Basis Expansion Analysis for Time Series) is a deep,
fully-connected architecture built from stacked residual blocks; each block
outputs a *backcast* (explains the input) and a *forecast* (extrapolates it).
Stacks can be **generic** (unconstrained basis, learns whatever functions help)
or **interpretable** (constrained to trend + seasonality basis functions, more
explainable but less flexible). It is trained **globally**: one model sees all
~3,000 (Store, Dept) series at once and shares weights across them, unlike
per-series ARIMA/SARIMA which fit each series independently.

**Key architectural property vs. the classical notebooks:** N-BEATS (in its
original form, as used here — `neuralforecast`'s `NBEATS`, not the extended
`NBEATSx`) is **purely univariate** — it only looks at a window of the series'
own past values (`input_size`), with **no exogenous inputs** (no Temperature,
CPI, markdowns, holiday flags) and **no static covariates** (no Store Type/Size).
That's a deliberate architectural choice worth discussing: whatever calendar/
holiday signal it picks up, it has to infer purely from the shape of the sales
history itself. The TFT notebook is the counterpoint — same competition, same
DL family, but explicitly designed to fuse static + exogenous + historical
inputs via attention.

**Direct multi-horizon forecasting:** N-BEATS predicts all `h` future steps in
one forward pass (unlike ARIMA's recursive one-step-at-a-time rollout), so `h`
must be fixed *before* training (it sizes the output layer). CV here uses a
small `h=8` model (matching the shared 8-week fold convention) purely to screen
hyperparameters cheaply; the actual submission horizon is 39 weeks, and a
mismatch between the two (a model tuned at h=8 is not guaranteed to behave the
same at h=39, since direct multi-horizon models can have non-uniform error
growth across the forecast window) is treated explicitly below rather than
assumed away.

**Tracking:** Weights & Biases (the assignment explicitly allows this for
neural network architectures).

## Experiment design

Instead of one flat grid over every hyperparameter at once (expensive, and
hard to reason about which axis mattered), the search here is a **3-stage
coarse-to-fine sweep**, each stage conditioned on the previous stage's winner:

1. **Architecture family** (`input_size` × `stack_types`) — the highest-level
   modeling choice (how much history to look back, generic vs
   trend+seasonality basis).
2. **Block depth** (`n_blocks`) — given the winning architecture, how many
   residual blocks per stack.
3. **Optimizer/batch** (`learning_rate` × `batch_size`) — given the winning
   architecture + depth, how to train it.

Each stage uses the same cheap 8-week expanding-window CV (3 folds,
`max_steps=150`) so results are comparable across stages. This keeps total
training cost far below a single flat grid over all 5 axes at once, while
still covering every axis. After the 3 stages pick a combined `best_config`,
there's a dedicated **true-horizon holdout check** (Run 5) at the real
39-week submission horizon before committing to the final full-data fit —
CV at `h=8` is a cheap proxy, not a guarantee, for `h=39` performance.

## 0. Config & environment (Kaggle vs local)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_COMPETITION = 'walmart-recruiting-store-sales-forecasting'
ON_KAGGLE = KAGGLE_INPUT.exists()

if ON_KAGGLE:
    _kaggle_candidates = [
        KAGGLE_INPUT / KAGGLE_COMPETITION,
        KAGGLE_INPUT / 'competitions' / KAGGLE_COMPETITION,
    ]
    RAW_DIR = next((c for c in _kaggle_candidates if c.exists()), _kaggle_candidates[0])
    WORKING_DIR = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path.cwd().parent
    RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
    WORKING_DIR = PROJECT_ROOT
    load_dotenv(PROJECT_ROOT / '.env')

def _resolve(stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        p = RAW_DIR / name
        if p.exists():
            return p
    return RAW_DIR / f'{stem}.csv'

TRAIN_CSV = _resolve('train')
TEST_CSV = _resolve('test')
FEATURES_CSV = _resolve('features')
STORES_CSV = _resolve('stores')

RANDOM_SEED = 42
TARGET = 'Weekly_Sales'
HOLIDAY_WEIGHT = 5
NON_HOLIDAY_WEIGHT = 1

if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for key in ('MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD', 'WANDB_API_KEY'):
            try:
                os.environ.setdefault(key, client.get_secret(key))
            except Exception:
                pass
    except Exception:
        pass

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

print('On Kaggle:', ON_KAGGLE, '| raw data dir:', RAW_DIR)

## 1. Data loading helpers

In [ ]:
import pandas as pd

def _read_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})

def load_stores():
    return pd.read_csv(STORES_CSV)

def load_features():
    df = pd.read_csv(FEATURES_CSV, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    df = df.sort_values(['Store', 'Date'])
    for col in ('CPI', 'Unemployment'):
        df[col] = df.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
    return df.reset_index(drop=True)

def load_raw(split):
    path = TRAIN_CSV if split == 'train' else TEST_CSV
    df = pd.read_csv(path, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    return df

def load_merged(split='train'):
    base = load_raw(split)
    stores = load_stores()
    feats = load_features().drop(columns=['IsHoliday'])
    df = base.merge(stores, on='Store', how='left')
    df = df.merge(feats, on=['Store', 'Date'], how='left')
    df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    return df

## 2. Metric (WMAE) & cross-validation helpers

In [ ]:
def weights_from_holiday(is_holiday):
    is_holiday = np.asarray(is_holiday).astype(bool)
    return np.where(is_holiday, HOLIDAY_WEIGHT, NON_HOLIDAY_WEIGHT).astype(float)

def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = weights_from_holiday(is_holiday)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

In [ ]:
def _sorted_unique_dates(dates):
    return np.sort(pd.to_datetime(dates).unique())

def time_holdout(df, weeks=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    if weeks >= len(uniq):
        raise ValueError(f'weeks={weeks} >= number of distinct weeks {len(uniq)}')
    cutoff = uniq[-weeks]
    d = pd.to_datetime(df[date_col]).to_numpy()
    train_idx = np.where(d < cutoff)[0]
    val_idx = np.where(d >= cutoff)[0]
    return train_idx, val_idx

def expanding_splits(df, n_splits=3, horizon=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    needed = horizon * n_splits
    if needed >= len(uniq):
        raise ValueError(f'Need > {needed} distinct weeks for {n_splits} folds of {horizon}; have {len(uniq)}.')
    d = pd.to_datetime(df[date_col]).to_numpy()
    for k in range(n_splits):
        end_offset = needed - k * horizon
        start_offset = end_offset - horizon
        val_start = uniq[-end_offset]
        val_end = uniq[-start_offset] if start_offset > 0 else None
        train_idx = np.where(d < val_start)[0]
        if val_end is None:
            val_idx = np.where(d >= val_start)[0]
        else:
            val_idx = np.where((d >= val_start) & (d < val_end))[0]
        yield train_idx, val_idx

## 3. Global-model (neuralforecast) shared helpers

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

NF_FREQ = 'W-FRI'

def make_unique_id(df):
    return df['Store'].astype(str) + '_' + df['Dept'].astype(str)

def cold_start_fallback_table(long_train):
    """Per-unique_id recent mean, plus a global mean for series never seen in training."""
    per_id = long_train.groupby('unique_id')['y'].apply(lambda s: s.tail(8).mean())
    global_mean = float(long_train['y'].mean())
    return per_id, global_mean

def fill_cold_start(pred_df, value_col, fallback_per_id, global_mean):
    missing = pred_df[value_col].isna()
    if missing.any():
        pred_df.loc[missing, value_col] = pred_df.loc[missing, 'unique_id'].map(fallback_per_id)
        pred_df[value_col] = pred_df[value_col].fillna(global_mean)
    return pred_df

## 4. Pipeline wrapper (raw-test-ready, wraps neuralforecast)

Moved ahead of the search stages (vs. the original single-shot version) since
every stage below now actually trains through this wrapper rather than just
logging config notes. `n_blocks` and `batch_size` are new constructor params
so Stages B and C can tune them without editing the class.

In [ ]:
def build_long_panel(df, value_col='Weekly_Sales'):
    out = pd.DataFrame({
        'unique_id': make_unique_id(df),
        'ds': pd.to_datetime(df['Date']),
        'y': df[value_col].to_numpy(dtype=float),
    })
    filled = []
    n_with_gaps = 0
    for uid, g in out.groupby('unique_id'):
        g = g.sort_values('ds').drop_duplicates('ds')
        full_idx = pd.date_range(g['ds'].min(), g['ds'].max(), freq=NF_FREQ)
        if len(full_idx) != len(g):
            n_with_gaps += 1
        s = g.set_index('ds')['y'].reindex(full_idx).interpolate(limit=4).ffill().bfill()
        filled.append(pd.DataFrame({'unique_id': uid, 'ds': full_idx, 'y': s.to_numpy()}))
    long_df = pd.concat(filled, ignore_index=True)
    return long_df, n_with_gaps


from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

class NBEATSPipeline(BaseEstimator, RegressorMixin):
    """sklearn-style wrapper so N-BEATS follows the same fit(raw_train)/predict(raw_test)
    contract as every other model in this repo. `horizon` must cover the longest
    gap (in weeks) between the end of training and the furthest requested date --
    fixed per assignment note above (direct multi-horizon architecture)."""

    def __init__(self, horizon=39, input_size=52, stack_types=None, n_blocks=None,
                 learning_rate=1e-3, batch_size=32, max_steps=300, random_seed=42):
        self.horizon = horizon
        self.input_size = input_size
        self.stack_types = stack_types or ['identity', 'identity', 'identity']
        self.n_blocks = n_blocks or [1] * len(self.stack_types)
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.max_steps = max_steps
        self.random_seed = random_seed

    def fit(self, X, y=None):
        df = X.copy()
        df['Weekly_Sales'] = np.asarray(y, dtype=float)
        self.long_train_, _ = build_long_panel(df)
        self.fallback_per_id_, self.global_mean_ = cold_start_fallback_table(self.long_train_)

        model = NBEATS(
            h=self.horizon, input_size=self.input_size, stack_types=self.stack_types,
            n_blocks=self.n_blocks, learning_rate=self.learning_rate, batch_size=self.batch_size,
            max_steps=self.max_steps, loss=MAE(), random_seed=self.random_seed,
        )
        self.nf_ = NeuralForecast(models=[model], freq=NF_FREQ)
        self.nf_.fit(df=self.long_train_)
        self.model_col_ = 'NBEATS'
        return self

    def predict(self, X):
        req = pd.DataFrame({'unique_id': make_unique_id(X), 'ds': pd.to_datetime(X['Date'])})
        fcst = self.nf_.predict()
        merged = req.merge(fcst[['unique_id', 'ds', self.model_col_]], on=['unique_id', 'ds'], how='left')
        merged = fill_cold_start(merged, self.model_col_, self.fallback_per_id_, self.global_mean_)
        return merged[self.model_col_].to_numpy()

## 5. Weights & Biases setup

In [ ]:
import wandb
import numpy as np, pandas as pd
import itertools

MODEL_NAME = 'NBEATS'
WANDB_PROJECT = 'walmart-sales-forecasting'
wandb.login()

## Load raw data

In [ ]:
raw_train = load_raw('train')
raw_test  = load_raw('test')
TEST_HORIZON = raw_test['Date'].nunique()
print(raw_train.shape, raw_test.shape, '| forecast horizon:', TEST_HORIZON, 'weeks')

## Run 1 — Cleaning
Build the long `unique_id / ds / y` panel neuralforecast expects, reindexing
each series to a continuous weekly frequency (small gaps interpolated) — the
same gap-handling idea as the classical notebooks, just reused here for a
different data shape.

In [ ]:
long_train, n_with_gaps = build_long_panel(raw_train)

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='cleaning', name=f'{MODEL_NAME}_Cleaning')
wandb.log({
    'n_series': long_train['unique_id'].nunique(),
    'n_series_with_gaps': n_with_gaps,
    'n_train_rows': len(raw_train),
    'n_negative_sales': int((raw_train.Weekly_Sales < 0).sum()),
})
run.finish()
print(long_train['unique_id'].nunique(), 'series,', n_with_gaps, 'with gaps (interpolated)')

## Run 2 — Architecture Search (Stage A: `input_size` x `stack_types`)
The highest-level modeling decision: how much history to look back, and
whether to use an unconstrained ("identity") basis or a
trend+seasonality-constrained ("interpretable") basis. `n_blocks` and the
optimizer are held fixed here so this stage isolates the architecture-family
effect. 4 configs x 3 expanding-window folds (`h=8`, `max_steps=150`).

In [ ]:
candidate_input_sizes = [26, 52]
candidate_stack_types = [
    ['identity', 'identity', 'identity'],
    ['trend', 'seasonality', 'identity'],
]
FIXED_LR = 1e-3
FIXED_BATCH = 32

stage_a_configs = [
    {'input_size': i, 'stack_types': s}
    for i, s in itertools.product(candidate_input_sizes, candidate_stack_types)
]
print(len(stage_a_configs), 'Stage A (architecture) configs to try')

stage_a_results = []
for i, cfg in enumerate(stage_a_configs):
    run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='architecture_search',
                      name=f'{MODEL_NAME}_ArchitectureSearch_trial{i}', config=cfg, reinit=True)
    fold_scores = []
    for k, (tr_idx, va_idx) in enumerate(expanding_splits(raw_train, n_splits=3, horizon=8)):
        tr, va = raw_train.iloc[tr_idx], raw_train.iloc[va_idx]
        pipe = NBEATSPipeline(horizon=8, max_steps=150, n_blocks=[1] * len(cfg['stack_types']),
                               learning_rate=FIXED_LR, batch_size=FIXED_BATCH, **cfg)
        pipe.fit(tr, tr['Weekly_Sales'])
        pred = pipe.predict(va)
        score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
        fold_scores.append(score)
        wandb.log({'fold': k, 'fold_wmae': score})
    cv_mean = float(np.mean(fold_scores))
    wandb.log({'cv_wmae_mean': cv_mean, 'cv_wmae_std': float(np.std(fold_scores))})
    run.finish()
    stage_a_results.append({'config': cfg, 'cv_wmae_mean': cv_mean})
    print(f'[Stage A] config {i} {cfg}: CV WMAE={cv_mean:,.1f}')

best_stage_a = min(stage_a_results, key=lambda t: t['cv_wmae_mean'])['config']
print('Stage A winner:', best_stage_a)

## Run 3 — Block-Depth Search (Stage B: `n_blocks`)
Given the winning architecture family from Stage A, sweep how many residual
blocks per stack (1 vs 2) — more blocks means more capacity per stack, at the
cost of more parameters and training time. 2 configs x 3 folds.

In [ ]:
n_stacks = len(best_stage_a['stack_types'])
candidate_n_blocks = [[1] * n_stacks, [2] * n_stacks]

stage_b_configs = [{'n_blocks': nb} for nb in candidate_n_blocks]
print(len(stage_b_configs), 'Stage B (block depth) configs to try')

stage_b_results = []
for i, cfg in enumerate(stage_b_configs):
    run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='block_search',
                      name=f'{MODEL_NAME}_BlockSearch_trial{i}', config={**best_stage_a, **cfg}, reinit=True)
    fold_scores = []
    for k, (tr_idx, va_idx) in enumerate(expanding_splits(raw_train, n_splits=3, horizon=8)):
        tr, va = raw_train.iloc[tr_idx], raw_train.iloc[va_idx]
        pipe = NBEATSPipeline(horizon=8, max_steps=150, learning_rate=FIXED_LR,
                               batch_size=FIXED_BATCH, **best_stage_a, **cfg)
        pipe.fit(tr, tr['Weekly_Sales'])
        pred = pipe.predict(va)
        score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
        fold_scores.append(score)
        wandb.log({'fold': k, 'fold_wmae': score})
    cv_mean = float(np.mean(fold_scores))
    wandb.log({'cv_wmae_mean': cv_mean, 'cv_wmae_std': float(np.std(fold_scores))})
    run.finish()
    stage_b_results.append({'config': cfg, 'cv_wmae_mean': cv_mean})
    print(f'[Stage B] config {i} {cfg}: CV WMAE={cv_mean:,.1f}')

best_stage_b = min(stage_b_results, key=lambda t: t['cv_wmae_mean'])['config']
print('Stage B winner:', best_stage_b)

## Run 4 — Optimizer/Batch Search (Stage C: `learning_rate` x `batch_size`)
Given the winning architecture + depth, sweep learning rate and batch size.
4 configs x 3 folds. The combined `best_config` after this stage is the one
carried into the true-horizon holdout check and final fit below.

In [ ]:
candidate_learning_rates = [1e-3, 5e-4]
candidate_batch_sizes = [64, 128]

stage_c_configs = [
    {'learning_rate': lr, 'batch_size': b}
    for lr, b in itertools.product(candidate_learning_rates, candidate_batch_sizes)
]
print(len(stage_c_configs), 'Stage C (optimizer) configs to try')

stage_c_results = []
for i, cfg in enumerate(stage_c_configs):
    run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='optimizer_search',
                      name=f'{MODEL_NAME}_OptimizerSearch_trial{i}',
                      config={**best_stage_a, **best_stage_b, **cfg}, reinit=True)
    fold_scores = []
    for k, (tr_idx, va_idx) in enumerate(expanding_splits(raw_train, n_splits=3, horizon=8)):
        tr, va = raw_train.iloc[tr_idx], raw_train.iloc[va_idx]
        pipe = NBEATSPipeline(horizon=8, max_steps=150, **best_stage_a, **best_stage_b, **cfg)
        pipe.fit(tr, tr['Weekly_Sales'])
        pred = pipe.predict(va)
        score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
        fold_scores.append(score)
        wandb.log({'fold': k, 'fold_wmae': score})
    cv_mean = float(np.mean(fold_scores))
    wandb.log({'cv_wmae_mean': cv_mean, 'cv_wmae_std': float(np.std(fold_scores))})
    run.finish()
    stage_c_results.append({'config': cfg, 'cv_wmae_mean': cv_mean})
    print(f'[Stage C] config {i} {cfg}: CV WMAE={cv_mean:,.1f}')

best_stage_c = min(stage_c_results, key=lambda t: t['cv_wmae_mean'])['config']
best_config = {**best_stage_a, **best_stage_b, **best_stage_c}
print('Stage C winner:', best_stage_c)
print('Combined best config across all 3 stages:', best_config)

## Results summary
Quick visual of how much each stage improved the best CV WMAE — useful as-is
for the README/report section on this architecture.

In [ ]:
import matplotlib.pyplot as plt

stage_bests = {
    'Stage A\n(architecture)': min(r['cv_wmae_mean'] for r in stage_a_results),
    'Stage B\n(block depth)': min(r['cv_wmae_mean'] for r in stage_b_results),
    'Stage C\n(optimizer)': min(r['cv_wmae_mean'] for r in stage_c_results),
}

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(stage_bests.keys()), list(stage_bests.values()), color=['#4C72B0', '#55A868', '#C44E52'])
ax.set_ylabel('Best CV WMAE (8-week expanding folds)')
ax.set_title(f'{MODEL_NAME} -- best CV WMAE after each search stage')
for i, v in enumerate(stage_bests.values()):
    ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig(str(Path(WORKING_DIR) / f'{MODEL_NAME}_stage_wmae.png'), dpi=120)
plt.show()

## Run 5 — True-horizon holdout check
CV above used `h=8` purely to keep the 3-stage search cheap. But the actual
Kaggle submission needs `h=39` (the full test period), and direct
multi-horizon models don't necessarily generalize their `h=8` accuracy out to
`h=39` — error can grow unevenly across the forecast window. So before
committing to the expensive full-data final fit, this holds out the *last
39 weeks* of `raw_train` and evaluates the combined best config at the real
submission horizon. This is the number that should actually predict Kaggle
leaderboard performance, not the Stage A/B/C CV means above.

In [ ]:
run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='holdout',
                  name=f'{MODEL_NAME}_Holdout_TrueHorizon', config={**best_config, 'horizon': TEST_HORIZON})

holdout_tr, holdout_va = time_holdout(raw_train, weeks=TEST_HORIZON)
p = NBEATSPipeline(horizon=TEST_HORIZON, max_steps=400, **best_config)
p.fit(raw_train.iloc[holdout_tr], raw_train.iloc[holdout_tr]['Weekly_Sales'])
hv = raw_train.iloc[holdout_va]
holdout_wmae = wmae(hv['Weekly_Sales'], p.predict(hv), hv['IsHoliday'])
wandb.log({'holdout_wmae_true_horizon': holdout_wmae})
run.finish()
print(f'True-horizon ({TEST_HORIZON}-week) holdout WMAE:', holdout_wmae)

## Run 6 — Final fit + save Pipeline
Refit `best_config` on the **full** raw_train with `horizon=TEST_HORIZON`
(the actual Kaggle test period, computed once from `raw_test` above -- never
a leftover value from an earlier search loop) and save the fitted Pipeline
(`neuralforecast`'s own `nf.save(...)`, plus a pickle of the wrapper config
so it can be reloaded and called exactly like the other notebooks' pipelines).
Wandb doesn't have the same "Model Registry" concept the assignment wants for
the *overall* best model -- if this architecture wins, log/register it via
MLflow's generic `pyfunc` flavor at that point (see note below).

In [ ]:
run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='final',
                  name=f'{MODEL_NAME}_Final', config={**best_config, 'horizon': TEST_HORIZON})

final_pipe = NBEATSPipeline(horizon=TEST_HORIZON, max_steps=400, **best_config)
final_pipe.fit(raw_train, raw_train['Weekly_Sales'])

import pickle, pathlib
out_dir = pathlib.Path(WORKING_DIR) / 'models' / MODEL_NAME
out_dir.mkdir(parents=True, exist_ok=True)
final_pipe.nf_.save(path=str(out_dir / 'nf_model'), overwrite=True)
with open(out_dir / 'pipeline_wrapper.pkl', 'wb') as f:
    pickle.dump({'horizon': TEST_HORIZON, **best_config}, f)
wandb.log_artifact(str(out_dir), name=f'{MODEL_NAME}_pipeline', type='model')
wandb.log({'held_out_wmae_before_full_refit': holdout_wmae})
run.finish()
print('saved pipeline to', out_dir)

## Run 7 — Sanity check (forecast window actually covers the test period)
A cheap guard against the single most common bug with this kind of pipeline:
predicting from a model whose forecast window doesn't actually reach the
Kaggle test dates (e.g. because it was fit on a truncated split, or `horizon`
didn't match the true test length). Reload the just-saved pipeline and check
its forecast range against `raw_test`'s date range before trusting it for
`model_inference.ipynb`.

In [ ]:
reloaded_nf = NeuralForecast.load(path=str(out_dir / 'nf_model'))
sanity_fcst = reloaded_nf.predict()
print('reloaded forecast date range:', sanity_fcst['ds'].min(), '->', sanity_fcst['ds'].max())
print('kaggle test date range:      ', raw_test['Date'].min(), '->', raw_test['Date'].max())
assert sanity_fcst['ds'].max() >= raw_test['Date'].max(), (
    'Forecast horizon does not cover the full Kaggle test period -- '
    'check that horizon=TEST_HORIZON was used and the pipeline was fit on the FULL raw_train, '
    'not a truncated holdout split.'
)
print('OK: forecast window covers the Kaggle test period.')

> **If N-BEATS turns out to be the overall best model:** wrap `final_pipe`
> (or reload it via `NeuralForecast.load(path=...)` + the pickled wrapper
> config) in a `mlflow.pyfunc.PythonModel` and log/register that with
> `mlflow.pyfunc.log_model(...)` + `mlflow.register_model(...)` so
> `model_inference.ipynb` can still load it the same way as the sklearn-based
> Pipelines from the Model Registry.